# Creating baseline model for forming portfolio

In [1]:
%pip install yfinance

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import yfinance as yf

In [2]:
def get_close_prices(tickers, target_date):
    data = yf.download(
        tickers,
        start=target_date,
        end=target_date + pd.Timedelta(days=14),
        interval="1d",
        auto_adjust=False,
        progress=False,
    )

    if data is None or data.empty:
        raise ValueError("Не удалось получить цены.")

    return data["Adj Close"].iloc[0]

# Testing tickers' price, dividends, portfolio distribution

In [15]:
tickers = ['AAPL', 'MSFT', 'AMZN', 'JPM', 'XOM', 'JNJ', 'WMT', 'CAT', 'NEE', 'KO',
                 'UPS', 'NVDA']
target_date = pd.Timestamp("2026-01-05")

data = yf.download(
    tickers,
    start=target_date,
    end=target_date + pd.Timedelta(days=1),
    interval="1d",
    auto_adjust=False,
    progress=False,
)

close_prices = data["Close"].iloc[0] # type: ignore
print(close_prices)

Ticker
AAPL    267.260010
AMZN    233.059998
CAT     616.099976
JNJ     204.309998
JPM     334.040009
KO       67.940002
MSFT    472.850006
NEE      81.320000
NVDA    188.119995
UPS     102.000000
WMT     112.709999
XOM     125.360001
Name: 2026-01-05 00:00:00, dtype: float64


In [16]:
average_close = close_prices.mean()
variance = close_prices.std()

print(f"Средняя цена закрытия: {average_close:.2f}")
print(f"Дисперсия: {variance:.2f}")


Средняя цена закрытия: 233.76
Дисперсия: 168.41


In [11]:
target_date = pd.Timestamp("2020-05-05")

data = yf.download(
    tickers,
    start=target_date,
    end=target_date + pd.Timedelta(days=1),
    interval="1d",
    auto_adjust=False,
    progress=False,
)

close_prices = data["Close"].iloc[0] # type: ignore
print(close_prices)

Ticker
AAPL     74.389999
AMZN    115.889999
CAT     108.910004
JNJ     149.500000
JPM      92.000000
KO       45.400002
MSFT    180.759995
NEE      57.525002
NVDA      7.343500
UPS      92.709999
WMT      41.576668
XOM      44.830002
Name: 2020-05-05 00:00:00, dtype: float64


In [12]:
average_close = close_prices.mean()
variance = close_prices.std()

print(f"Средняя цена закрытия: {average_close:.2f}")
print(f"Дисперсия: {variance:.2f}")

Средняя цена закрытия: 84.24
Дисперсия: 49.47


# 1 test step of baseline model cycle

**Initializing capital**

In [3]:
initial_capital = 100_000

**Close prices of each portfolios' position**

In [14]:
tickers = ['AAPL', 'MSFT', 'AMZN', 'JPM', 'XOM', 'JNJ', 'WMT', 'CAT', 'NEE', 'KO',
                 'UPS', 'NVDA']
target_date = pd.Timestamp("2020-01-02")

close_prices = get_close_prices(tickers, target_date)

print(close_prices)

Ticker
AAPL     75.087502
AMZN     94.900497
CAT     150.529999
JNJ     145.970001
JPM     141.089996
KO       54.990002
MSFT    160.619995
NEE      59.654999
NVDA      5.997750
UPS     116.790001
WMT      39.646667
XOM      70.900002
Name: 2020-01-02 00:00:00, dtype: float64


**Calculating number of each position**

In [10]:
number_of_positions = initial_capital/(len(tickers) * close_prices)
number_of_positions

Ticker
AAPL     110.981630
AMZN      87.811271
CAT       55.359951
JNJ       57.089356
JPM       59.063956
KO       151.542700
MSFT      51.882291
NEE      139.692121
NVDA    1389.409963
UPS       71.353140
WMT      210.190007
XOM      117.536434
Name: 2020-01-02 00:00:00, dtype: float64

**Calculating capital in a quarter**

In [17]:
target_date = pd.Timestamp("2020-04-01")

close_prices = get_close_prices(tickers, target_date)

capital = sum(close_prices * number_of_positions)
capital

82757.5946734973

# The work cycle for 2 years

In [3]:
initial_capital = 100_000
capital = initial_capital
tickers = ['AAPL', 'MSFT', 'AMZN', 'JPM', 'XOM', 'JNJ', 'WMT', 'CAT', 'NEE', 'KO',
                 'UPS', 'NVDA']
start_date = pd.Timestamp("2020-01-02")

for months in range(0, 25, 3):
    # forming pd.Series of stock prices
    target_date = start_date + pd.DateOffset(months=months)
    print(f"\nПлановая дата: {target_date.date()}")
    close_prices = get_close_prices(tickers, target_date)
    print(close_prices)
    print()

    # recalculating portfolio capital
    if months > 0:
        capital = sum(close_prices * number_of_positions)
        print(capital)
    else:
        print(initial_capital)
    print()

    # forming number of portfolio positions
    number_of_positions = capital/(len(tickers) * close_prices)
    print(number_of_positions)
    print()

print(initial_capital, capital, sep=',')


Плановая дата: 2020-01-02
Ticker
AAPL     72.271500
AMZN     94.900497
CAT     132.067871
JNJ     121.338524
JPM     117.899216
KO       45.141266
MSFT    151.544266
NEE      50.335987
NVDA      5.963803
UPS      87.781105
WMT      36.203281
XOM      52.606407
Name: 2020-01-02 00:00:00, dtype: float64

100000

Ticker
AAPL     115.305942
AMZN      87.811271
CAT       63.098869
JNJ       68.678381
JPM       70.681839
KO       184.605663
MSFT      54.989434
NEE      165.554185
NVDA    1397.318611
UPS       94.933110
WMT      230.181713
XOM      158.409095
Name: 2020-01-02 00:00:00, dtype: float64


Плановая дата: 2020-04-02
Ticker
AAPL     59.076008
AMZN     95.941498
CAT     103.140572
JNJ     111.387589
JPM      73.595543
KO       36.394947
MSFT    146.887283
NEE      48.018242
NVDA      6.354400
UPS      70.623169
WMT      36.274815
XOM      30.406351
Name: 2020-04-02 00:00:00, dtype: float64

86092.02353877752

Ticker
AAPL     121.442453
AMZN      74.778229
CAT       69.558809
JNJ   